# Lesson 3: System Prompts

Shape Gemini's role/tone with `system_instruction` (Claude's `system=`).

In [ ]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(override=True)

client = genai.Client()
model = "gemini-3.6-flash"

Helpers. `chat` only adds `system_instruction` when you pass `system`. Retries on 503 (model busy).

In [ ]:
import time
from google.genai.errors import ServerError


def add_user_message(messages, text):
    messages.append({"role": "user", "parts": [{"text": text}]})


def add_assistant_message(messages, text):
    messages.append({"role": "model", "parts": [{"text": text}]})


def generate_with_retry(**params):
    for attempt in range(5):
        try:
            return client.models.generate_content(**params)
        except ServerError:
            if attempt == 4:
                raise
            time.sleep(2 ** attempt)


def chat(messages, system=None):
    params = {
        "model": model,
        "contents": messages,
    }

    config_kwargs = {"max_output_tokens": 1000}
    if system:
        config_kwargs["system_instruction"] = system

    params["config"] = types.GenerateContentConfig(**config_kwargs)

    return generate_with_retry(**params).text

Same question, no system prompt — usually dumps the full answer.

In [ ]:
student_question = "How do I solve 5x + 2 = 3 for x?"

messages = []
add_user_message(messages, student_question)

answer = chat(messages)
print(answer)

Same question as a patient tutor — hints, not the solution.

In [ ]:
system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

messages = []
add_user_message(messages, student_question)

answer = chat(messages, system=system_prompt)
print(answer)

Same system prompt on native chat (`client.chats.create`).

In [ ]:
tutor_chat = client.chats.create(
    model=model,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        max_output_tokens=1000,
    ),
)

for attempt in range(5):
    try:
        response = tutor_chat.send_message(student_question)
        print(response.text)
        break
    except ServerError:
        if attempt == 4:
            raise
        time.sleep(2 ** attempt)